# Task 2a: Corruption Classifier Optuna Search & Full Retraining

Hyperparameter tuning and full-schedule retraining for the Task 2a CNN classifier as specified in the assignment.

### Mathematical Formulation
$$p = f(\tilde{x}) = [p_{clean},\, p_{salt},\, p_{blur},\, p_{occlusion}], \quad \hat{k} = \arg\max_{k}\, p_k$$
$$\mathcal{L}_{CE} = -\sum_{k=1}^{4} y_k \log p_k$$

### Optuna Search Space Requirements
As required by the assignment, Optuna tunes:
1. **Learning Rate** ($\text{lr} \in [10^{-4}, 10^{-2}]$ log scale)
2. **Batch Size** ($B \in \{32, 64, 128\}$)
3. **Convolutional Channel Configuration** ($\text{base\_channels} \in \{16, 32, 64\}$)
4. **Dropout Rate** ($\text{dropout} \in [0.0, 0.5]$)
5. **Weight Decay** ($\text{weight\_decay} \in [10^{-5}, 10^{-3}]$ log scale)

### Retraining:
The winning configuration is retrained for `FULL_TRAIN_EPOCHS = 35` to produce `task2a_classifier_final_best.pt` for downstream hard routing and soft MoE initialization.

In [ ]:
# 1. Config
N_TRIALS = 20
OPTUNA_EPOCHS = 15
FULL_TRAIN_EPOCHS = 35
STUDY_NAME = "task2a_classifier"
STORAGE_DB = "sqlite:///task2a_study.db"
SEED = 42

In [ ]:
# 2. Imports & Setup
import sys
from pathlib import Path
import yaml
import torch
from torch.utils.data import DataLoader
import optuna

RESEARCH_ROOT = Path("..").resolve()
if str(RESEARCH_ROOT) not in sys.path:
    sys.path.insert(0, str(RESEARCH_ROOT))

from constants import CONFIGS_ROOT
from src.device_utils import get_device, device_report
from src.datasets import PetDataset
from src.models_classifier import CorruptionClassifier
from src.train_loop import train_one_epoch_classifier, validate_classifier, set_seed
from src.checkpoint_utils import create_checkpoint_dict, save_checkpoint

set_seed(SEED)
device = get_device()

In [ ]:
# 3. Data Loading
train_dataset = PetDataset(mode="train", corruption_mode="label", seed=SEED)
val_dataset = PetDataset(mode="val", corruption_mode="label", seed=SEED)

In [ ]:
# 4. Optuna Objective
def objective(trial: optuna.Trial) -> float:
    lr = trial.suggest_float("lr", 1e-4, 1e-2, log=True)
    batch_size = trial.suggest_categorical("batch_size", [32, 64, 128])
    base_channels = trial.suggest_categorical("base_channels", [16, 32, 64])
    dropout = trial.suggest_float("dropout", 0.0, 0.5)
    weight_decay = trial.suggest_float("weight_decay", 1e-5, 1e-3, log=True)

    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=2)
    val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False, num_workers=2)

    model = CorruptionClassifier(
        base_channels=base_channels,
        dropout=dropout,
        num_classes=4
    ).to(device)

    optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)
    best_acc = 0.0

    for epoch in range(1, OPTUNA_EPOCHS + 1):
        train_one_epoch_classifier(model, train_loader, optimizer, device)
        val_res = validate_classifier(model, val_loader, device)
        acc = val_res["val_acc"]
        if acc > best_acc:
            best_acc = acc
        trial.report(acc, step=epoch)
        if trial.should_prune():
            raise optuna.TrialPruned()

    return best_acc

In [ ]:
# 5. Run Optimization
study = optuna.create_study(
    study_name=STUDY_NAME,
    storage=STORAGE_DB,
    load_if_exists=True,
    direction="maximize",
    pruner=optuna.pruners.MedianPruner()
)
study.optimize(objective, n_trials=N_TRIALS)

print(f"Best Trial: #{study.best_trial.number} with Accuracy: {study.best_value:.2%}")
print(study.best_params)

In [ ]:
# 6. Save Best Config & Retrain Winning Model on Full Schedule
best_config_path = CONFIGS_ROOT / "task2a_best_config.yaml"
with open(best_config_path, "w") as f:
    yaml.dump({
        "task": "task2a_corruption_classifier",
        "best_trial_number": int(study.best_trial.number),
        "best_accuracy": float(study.best_value),
        "params": study.best_params,
    }, f, indent=2)
print(f"[Config] Saved best classifier hyperparameters to {best_config_path}")

# Retrain winning configuration on full schedule
print(f"\n--- Retraining Winning Classifier ({FULL_TRAIN_EPOCHS} Epochs) ---")
p = study.best_params
train_loader = DataLoader(train_dataset, batch_size=p["batch_size"], shuffle=True, num_workers=2)
val_loader = DataLoader(val_dataset, batch_size=p["batch_size"], shuffle=False, num_workers=2)

winner_model = CorruptionClassifier(
    base_channels=p["base_channels"],
    dropout=p["dropout"],
    num_classes=4
).to(device)

optimizer = torch.optim.Adam(winner_model.parameters(), lr=p["lr"], weight_decay=p["weight_decay"])
scaler = torch.amp.GradScaler('cuda') if device.type == "cuda" else None
best_val_acc = 0.0
checkpoint_dir = RESEARCH_ROOT / "checkpoints"

for epoch in range(1, FULL_TRAIN_EPOCHS + 1):
    tr = train_one_epoch_classifier(winner_model, train_loader, optimizer, device, scaler=scaler)
    vl = validate_classifier(winner_model, val_loader, device)
    val_acc = vl["val_acc"]
    is_best = val_acc > best_val_acc
    if is_best:
        best_val_acc = val_acc
    ckpt = create_checkpoint_dict(
        winner_model, optimizer, epoch, vl["val_loss"],
        optuna_trial_number=int(study.best_trial.number),
        random_seed=SEED,
        extra_metadata={**p, "val_acc": val_acc}
    )
    save_checkpoint(
        checkpoint_dir=checkpoint_dir,
        prefix="task2a_classifier_final",
        ckpt_dict=ckpt,
        is_best=is_best,
        task_name="task2a",
        phase_name="final",
        device_name=device_report()["device_name"]
    )
    print(f"Epoch [{epoch:02d}/{FULL_TRAIN_EPOCHS:02d}] Train Acc: {tr['train_acc']:.2%} | Val Acc: {val_acc:.2%} {'*Best*' if is_best else ''}")

print(f"Final retrained classifier best accuracy: {best_val_acc:.2%}")